
# Customer Churn Project — S3 Bucket + Validation Lambda Setup

This notebook provisions the ingestion layer for the churn prediction project:

1. An S3 bucket with a `raw/`, `processed/`, `output/` folder structure
2. A validation Lambda function that triggers automatically when a file lands in `raw/`
3. The S3 → Lambda event trigger wiring

**How to use this notebook**

- Run it against **Floci** (local AWS emulator) first to develop and test for free, with zero risk to any real AWS account.
- When you're ready for your final submission, change `USE_LOCAL_EMULATOR = False` in the config cell below, add your real AWS credentials, and re-run — the code is identical either way. This is exactly the kind of "AWS Free Tier" justification your report should mention.

This satisfies these assignment requirements: storing the original dataset in the cloud,
organizing raw/processed/output data, automatically processing newly uploaded datasets,
basic validation of uploaded data, and logs for monitoring/troubleshooting.


## 1. Configuration

In [ ]:
import io
import json
import logging
import os
import time
import zipfile

import boto3

USE_LOCAL_EMULATOR = False

BUCKET_NAME   = "churn-data-4736"   # must be globally unique on real AWS — add a suffix, e.g. "-yourname"
REGION        = "us-east-1"
LAMBDA_NAME   = "churn-data-validator"
ROLE_NAME     = "churn-lambda-execution-role"

logger = logging.getLogger(__name__)

if USE_LOCAL_EMULATOR:
    ENDPOINT_URL = "http://localhost:4566"   # Floci
    ACCESS_KEY   = "test"
    SECRET_KEY   = "test"
else:
    ENDPOINT_URL = None                      # real AWS resolves the endpoint itself
    ACCESS_KEY   = os.environ["AWS_ACCESS_KEY_ID"]      # never hardcode real credentials
    SECRET_KEY   = os.environ["AWS_SECRET_ACCESS_KEY"]

session = boto3.session.Session(
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
)

s3     = session.client("s3", endpoint_url=ENDPOINT_URL)
iam    = session.client("iam", endpoint_url=ENDPOINT_URL)
lam    = session.client("lambda", endpoint_url=ENDPOINT_URL)
logs   = session.client("logs", endpoint_url=ENDPOINT_URL)

print("Configured for:", "Floci (local)" if USE_LOCAL_EMULATOR else "Real AWS")

Configured for: Floci (local)


## 2. Create the S3 bucket and folder structure

In [ ]:
def create_bucket(name):
    try:
        if REGION == "us-east-1":
            s3.create_bucket(Bucket=name)
        else:
            s3.create_bucket(
                Bucket=name,
                CreateBucketConfiguration={"LocationConstraint": REGION},
            )
        print(f"Created bucket: {name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Bucket already exists: {name}")
    except Exception:
        logger.exception("Unexpected errors")
        raise

bucket_name = "floci-project-data"
create_bucket(bucket_name)

# "Folders" in S3 are just zero-byte objects with a trailing slash key.
for prefix in ["raw/", "raw/rejected/", "processed/", "output/"]:
    s3.put_object(Bucket=bucket_name, Key=prefix)
    print("Created prefix:", prefix)

Created bucket: floci-project-data
Created prefix: raw/
Created prefix: raw/rejected/
Created prefix: processed/
Created prefix: output/



## 3. Enable a basic security baseline

Two quick wins the assignment explicitly asks for ("apply appropriate security
practices"): block public access, and turn on versioning so accidental
overwrites/deletes in `processed/` or `output/` are recoverable.


In [ ]:
try:
    s3.put_public_access_block(
        Bucket=bucket_name,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True,
            "IgnorePublicAcls": True,
            "BlockPublicPolicy": True,
            "RestrictPublicBuckets": True,
        },
    )
    print("Public access blocked.")
except Exception:
    logger.exception("Public access block skipped/not supported here:")
    raise

try:
    s3.put_bucket_versioning(
        Bucket=bucket_name,
        VersioningConfiguration={"Status": "Enabled"},
    )
    print("Versioning enabled.")
except Exception:
    logger.exception("Versioning skipped/not supported here:")
    raise


Public access blocked.
Versioning enabled.


## 4. Write the Lambda validation function

In [9]:

lambda_source = r'''
"""
lambda_function.py
-------------------
Triggered by S3 ObjectCreated events on the `raw/` prefix of the churn
data bucket. Validates the uploaded CSV and routes it:

    raw/<file>.csv
        -> valid   -> processed/<file>.csv   (copy of the original)
        -> invalid -> raw/rejected/<file>.csv (moved, with a .json report)

All validation steps and outcomes are written to CloudWatch Logs so the
group can demonstrate monitoring/troubleshooting for the assignment.

Environment variables expected on the Lambda function:
    PROCESSED_PREFIX   default "processed/"
    REJECTED_PREFIX    default "raw/rejected/"
    REQUIRED_COLUMNS   comma-separated list, e.g.
                        "customerID,tenure,MonthlyCharges,TotalCharges,Churn"
"""

import csv
import io
import json
import logging
import os
import urllib.parse
from datetime import datetime, timezone

import boto3

logger = logging.getLogger()
logger.setLevel(logging.INFO)

s3 = boto3.client("s3")

PROCESSED_PREFIX = os.environ.get("PROCESSED_PREFIX", "processed/")
REJECTED_PREFIX = os.environ.get("REJECTED_PREFIX", "raw/rejected/")
REQUIRED_COLUMNS = [
    c.strip()
    for c in os.environ.get(
        "REQUIRED_COLUMNS",
        "customerID,tenure,MonthlyCharges,TotalCharges,Churn",
    ).split(",")
    if c.strip()
]

# Columns that must contain numeric values (used for the dtype check)
NUMERIC_COLUMNS = {"tenure", "MonthlyCharges", "TotalCharges"}


def validate_csv(body_bytes):
    """
    Runs basic data-quality checks on a CSV file's raw bytes.
    Returns (is_valid: bool, report: dict)
    """
    report = {
        "checked_at": datetime.now(timezone.utc).isoformat(),
        "row_count": 0,
        "missing_columns": [],
        "missing_value_counts": {},
        "type_errors": {},
        "issues": [],
    }

    try:
        text = body_bytes.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        report["issues"].append(f"File is not valid UTF-8 text: {exc}")
        return False, report

    reader = csv.DictReader(io.StringIO(text))
    fieldnames = reader.fieldnames or []

    missing_columns = [c for c in REQUIRED_COLUMNS if c not in fieldnames]
    report["missing_columns"] = missing_columns
    if missing_columns:
        report["issues"].append(f"Missing required columns: {missing_columns}")

    missing_counts = {c: 0 for c in REQUIRED_COLUMNS if c in fieldnames}
    type_error_counts = {c: 0 for c in NUMERIC_COLUMNS if c in fieldnames}
    row_count = 0

    for row in reader:
        row_count += 1
        for col in missing_counts:
            value = (row.get(col) or "").strip()
            if value == "":
                missing_counts[col] += 1

        for col in type_error_counts:
            value = (row.get(col) or "").strip()
            if value != "":
                try:
                    float(value)
                except ValueError:
                    type_error_counts[col] += 1

    report["row_count"] = row_count
    report["missing_value_counts"] = missing_counts
    report["type_errors"] = type_error_counts

    if row_count == 0:
        report["issues"].append("File contains no data rows.")

    total_missing = sum(missing_counts.values())
    total_type_errors = sum(type_error_counts.values())
    if total_missing > 0:
        report["issues"].append(f"{total_missing} missing values across required columns.")
    if total_type_errors > 0:
        report["issues"].append(f"{total_type_errors} non-numeric values in numeric columns.")

    is_valid = not missing_columns and row_count > 0 and total_type_errors == 0
    return is_valid, report


def lambda_handler(event, context):
    results = []

    for record in event.get("Records", []):
        bucket = record["s3"]["bucket"]["name"]
        key = urllib.parse.unquote_plus(record["s3"]["object"]["key"])
        file_name = key.split("/")[-1]

        logger.info("Processing s3://%s/%s", bucket, key)

        obj = s3.get_object(Bucket=bucket, Key=key)
        body = obj["Body"].read()

        is_valid, report = validate_csv(body)

        if is_valid:
            dest_key = f"{PROCESSED_PREFIX}{file_name}"
            s3.put_object(Bucket=bucket, Key=dest_key, Body=body)
            logger.info("VALID: copied to s3://%s/%s (%d rows)", bucket, dest_key, report["row_count"])
        else:
            dest_key = f"{REJECTED_PREFIX}{file_name}"
            report_key = f"{REJECTED_PREFIX}{file_name}.report.json"
            s3.put_object(Bucket=bucket, Key=dest_key, Body=body)
            s3.put_object(
                Bucket=bucket,
                Key=report_key,
                Body=json.dumps(report, indent=2).encode("utf-8"),
                ContentType="application/json",
            )
            logger.warning("INVALID: %s -> %s | issues=%s", key, dest_key, report["issues"])

        results.append({"source_key": key, "valid": is_valid, "destination": dest_key, "report": report})

    return {"statusCode": 200, "body": json.dumps(results, default=str)}

'''

with open("lambda_function.py", "w") as f:
    f.write(lambda_source)

print("Wrote lambda_function.py")


Wrote lambda_function.py


In [10]:

# Package it as the zip file Lambda's API expects
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("lambda_function.py")
zip_bytes = buf.getvalue()

with open("lambda_function.zip", "wb") as f:
    f.write(zip_bytes)

print(f"Packaged lambda_function.zip ({len(zip_bytes)} bytes)")


Packaged lambda_function.zip (1962 bytes)



## 5. Create an IAM execution role for the Lambda

The role needs:
- `AWSLambdaBasicExecutionRole`-style permissions to write CloudWatch Logs
- Read/write access limited to this one bucket (least privilege — another
  security practice worth citing in your report)


In [11]:

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

s3_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*",
        },
    ],
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
    )
    role_arn = role["Role"]["Arn"]
    print("Created role:", role_arn)
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print("Role already exists:", role_arn)

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="churn-lambda-s3-logs-access",
    PolicyDocument=json.dumps(s3_policy),
)
print("Attached inline policy.")

# Give the emulator/real IAM a moment for role propagation
time.sleep(3 if not USE_LOCAL_EMULATOR else 0.5)


Created role: arn:aws:iam::000000000000:role/churn-lambda-execution-role
Attached inline policy.


## 6. Create the Lambda function

In [12]:

env_vars = {
    "PROCESSED_PREFIX": "processed/",
    "REJECTED_PREFIX": "raw/rejected/",
    "REQUIRED_COLUMNS": "customerID,tenure,MonthlyCharges,TotalCharges,Churn",
}

try:
    fn = lam.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=30,
        MemorySize=256,
        Environment={"Variables": env_vars},
    )
    function_arn = fn["FunctionArn"]
    print("Created function:", function_arn)
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=zip_bytes)
    lam.update_function_configuration(
        FunctionName=LAMBDA_NAME,
        Environment={"Variables": env_vars},
    )
    function_arn = lam.get_function(FunctionName=LAMBDA_NAME)["Configuration"]["FunctionArn"]
    print("Updated existing function:", function_arn)

time.sleep(2)


Created function: arn:aws:lambda:us-east-1:000000000000:function:churn-data-validator



## 7. Wire up the S3 → Lambda trigger

Two things are required on real AWS (harmless to run against Floci too):
give S3 permission to invoke the function, then register the event
notification on the bucket for `ObjectCreated` events under `raw/`.


In [14]:

try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="AllowS3Invoke",
        Action="lambda:InvokeFunction",
        Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{bucket_name}",
    )
    print("Granted S3 invoke permission.")
except lam.exceptions.ResourceConflictException:
    print("Permission already granted.")

s3.put_bucket_notification_configuration(
    Bucket=BUCKET_NAME,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [
            {
                "LambdaFunctionArn": function_arn,
                "Events": ["s3:ObjectCreated:*"],
                "Filter": {
                    "Key": {
                        "FilterRules": [
                            {"Name": "prefix", "Value": "raw/"},
                            {"Name": "suffix", "Value": ".csv"},
                        ]
                    }
                },
            }
        ]
    },
)
print("S3 event trigger configured for raw/*.csv -> Lambda.")


Permission already granted.


NoSuchBucket: An error occurred (NoSuchBucket) when calling the PutBucketNotificationConfiguration operation: The specified bucket does not exist.

## 8. Test end-to-end: upload a sample file and check the result

In [ ]:

# A tiny sample — swap this for your real churn dataset before the final run
sample_csv = (
    "customerID,tenure,MonthlyCharges,TotalCharges,Churn\n"
    "0001,12,65.5,786.0,No\n"
    "0002,1,89.9,89.9,Yes\n"
    "0003,,70.0,,No\n"          # missing values -> should be rejected
)

s3.put_object(Bucket=BUCKET_NAME, Key="raw/sample_batch_1.csv", Body=sample_csv.encode("utf-8"))
print("Uploaded raw/sample_batch_1.csv")

time.sleep(3)  # give the trigger a moment to fire


In [ ]:

# Check where it ended up
for prefix in ["processed/", "raw/rejected/"]:
    resp = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"] != prefix]
    print(prefix, "->", keys)


In [ ]:

# Pull the CloudWatch logs to see the validation trail (for your monitoring evidence)
log_group = f"/aws/lambda/{LAMBDA_NAME}"
try:
    streams = logs.describe_log_streams(
        logGroupName=log_group, orderBy="LastEventTime", descending=True, limit=1
    )["logStreams"]
    if streams:
        events = logs.get_log_events(
            logGroupName=log_group, logStreamName=streams[0]["logStreamName"]
        )["events"]
        for e in events:
            print(e["message"].rstrip())
    else:
        print("No log streams yet — the function may not have run.")
except Exception:
    logger.exception("Could not read logs (this can happen if it hasn't invoked yet):")
    raise


Could not read logs (this can happen if it hasn't invoked yet): An error occurred (ResourceNotFoundException) when calling the DescribeLogStreams operation: The specified log group does not exist: /aws/lambda/churn-data-validator
Traceback (most recent call last):
  File "/tmp/ipykernel_164844/389355507.py", line 4, in <module>
    streams = logs.describe_log_streams(
              ~~~~~~~~~~~~~~~~~~~~~~~~~^
        logGroupName=log_group, orderBy="LastEventTime", descending=True, limit=1
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )["logStreams"]
    ^
  File "/home/user/git/customer-churn-aws-project/.venv/lib64/python3.14/site-packages/botocore/client.py", line 606, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/user/git/customer-churn-aws-project/.venv/lib64/python3.14/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kw

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the DescribeLogStreams operation: The specified log group does not exist: /aws/lambda/churn-data-validator


## 9. Notes for the report

- **Automatic processing**: the S3 event notification means no manual step is
  needed once a file lands in `raw/` — satisfies "automatically detect or
  process newly uploaded datasets."
- **Validation**: `lambda_function.py` checks required columns, missing
  values, and numeric-type violations, and writes a JSON report alongside
  any rejected file — satisfies both the validation and "identify missing
  values / incorrect data types" requirements.
- **Logging**: every invocation logs to CloudWatch (`/aws/lambda/churn-data-validator`),
  giving you the monitoring/troubleshooting trail the brief asks for. Take a
  screenshot of this log group for your report.
- **Security**: least-privilege IAM policy (scoped to this one bucket only),
  public access block, and versioning are quick, defensible justifications
  for your "appropriate security practices" section.
- **Cost control**: everything here — S3 standard storage in small volumes,
  one Lambda function invoked occasionally — sits comfortably inside AWS
  Free Tier limits. Mention in your report that you set a **AWS Budget alert**
  (Billing → Budgets) as an extra safeguard; that's a one-click console step
  worth a screenshot rather than API code.
- **Switching to real AWS**: set `USE_LOCAL_EMULATOR = False`, export real
  credentials as environment variables (never hardcode them — this is itself
  a security practice worth naming), give the bucket a globally-unique name,
  and rerun this notebook top to bottom.
